# E05 — previsão de atraso do primeiro desembolso

Piloto supervisionado de Ítalo Oliveira com dados públicos do Transferegov. A unidade é o instrumento discricionário e o alvo mede a ausência de primeiro desembolso em 365 dias após a assinatura. O horizonte é uma convenção operacional, não um prazo legal.

Apenas XGBoost e LightGBM são treinados nesta contribuição. Regressão Logística, KNN, Random Forest, Decision Tree, SVC e MLP ficam pendentes de integração. ObrasGov, PNCP, TSE, MUNIC, IBGE, SICONFI e dados fiscais não entram em X ou y.

In [1]:
from pathlib import Path
import json
import platform
import sys
import pandas as pd

from src.e05_transferegov_atraso import (
    GOLD_VERSION, LOAD_DATE, TARGET, FEATURE_COLUMNS, CATEGORICAL_FEATURES,
    NUMERIC_FEATURES, build_target, classification_metrics, diagnose_errors,
    load_frozen_tables, model_registry, select_predictors, temporal_split,
    tune_model, choose_threshold, frozen_paths, cv_folds, fit_baselines, permutation_sanity,
)

ROOT = Path.cwd()
OUT = ROOT / '.local' / 'e05_italo'
OUT.mkdir(parents=True, exist_ok=True)
print({'python': sys.version, 'platform': platform.platform(), 'gold_version': GOLD_VERSION, 'output': OUT.relative_to(ROOT).as_posix()})

{'python': '3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'gold_version': '65260a170ee18cd79501', 'output': '.local/e05_italo'}


## Versão congelada e inventário

A leitura usa exclusivamente a pasta da versão Gold congelada. O manifesto e os hashes são conferidos antes da construção do alvo. Bronze, Silver e Gold não são modificados.

In [2]:
instrumentos, desembolsos, manifesto = load_frozen_tables(ROOT / 'data')
assert manifesto['versao_gold'] == GOLD_VERSION
inventory = pd.DataFrame([
    {'tabela': 'silver/transferegov_instrumentos', 'camada': 'Silver', 'linhas': len(instrumentos), 'colunas': len(instrumentos.columns), 'chave': 'nr_convenio/id_proposta', 'duplicidades': int(instrumentos.nr_convenio.duplicated().sum())},
    {'tabela': 'silver/transferegov_desembolsos', 'camada': 'Silver', 'linhas': len(desembolsos), 'colunas': len(desembolsos.columns), 'chave': 'id_desembolso', 'duplicidades': int(desembolsos.id_desembolso.duplicated().sum())},
])
inventory.to_csv(OUT / 'inventario_transferegov.csv', index=False)
print(inventory.to_string(index=False))
print({'data_carga': '17/07/2026 06:30:47', 'instrumentos': len(instrumentos), 'desembolsos': len(desembolsos)})

                          tabela camada  linhas  colunas                   chave  duplicidades
silver/transferegov_instrumentos Silver    6182       34 nr_convenio/id_proposta             0
 silver/transferegov_desembolsos Silver    9411        5           id_desembolso             0
{'data_carga': '17/07/2026 06:30:47', 'instrumentos': 6182, 'desembolsos': 9411}


## Alvo, primeiro desembolso e censura

`risco_atraso_primeiro_desembolso_365d = 1` quando não há desembolso válido até o 365º dia. A classe 0 exige desembolso dentro da janela. Instrumentos sem 365 dias completos permanecem censurados e fora da modelagem. Desembolsos duplicados são deduplicados pela chave oficial e anomalias temporais ficam registradas no relatório.

In [3]:
target_frame = build_target(instrumentos, desembolsos, LOAD_DATE)
target_report = target_frame.attrs['report']
pd.DataFrame([target_report]).to_json(OUT / 'relatorio_alvo.json', orient='records', indent=2)
cohort_counts = target_frame.groupby('ano_ref').agg(
    instrumentos=('nr_convenio', 'size'),
    observaveis=('censurado', lambda s: int((~s).sum())),
    censurados=('censurado', 'sum'),
    positivos=(TARGET, lambda s: int((s == 1).sum())),
    negativos=(TARGET, lambda s: int((s == 0).sum())),
    municipios=('codigo_ibge', 'nunique'),
).reset_index()
cohort_counts.to_csv(OUT / 'coortes.csv', index=False)
print(target_report)
print(cohort_counts[cohort_counts.ano_ref.isin([2022, 2023, 2024])].to_string(index=False))

{'duplicate_disbursements': 0, 'unmatched_disbursements': 0, 'before_signature': 0, 'invalid_disbursement_dates': 0, 'invalid_disbursement_values': 0, 'instruments_with_first_disbursement': 4592, 'instruments_without_first_disbursement': 1590, 'unmatched_instruments': 1590, 'invalid_instrument_values': 4263, 'boundary_365': 7, 'boundary_gt_365': 5, 'reason_counts': {'desembolso_apos_365': 3383, 'sem_desembolso_ate_365': 1152, 'desembolso_ate_365': 1100, 'censurado_365_incompleto': 547}, 'observed': 5635, 'censored': 547, 'horizon_days': 365, 'data_carga': '2026-07-17'}
 ano_ref  instrumentos  observaveis  censurados  positivos  negativos  municipios
    2022           239          239           0        211         28         129
    2023           386          386           0        309         77         180
    2024           242          242           0        212         30         122


## Preditores e prevenção de leakage

Os preditores são conhecidos na assinatura: valores iniciais, modalidade, órgãos, emendas e durações originalmente pactuadas. Identificadores, desembolsos, resultados administrativos, situação final, CNPJ, textos e fontes externas são excluídos. O `ColumnTransformer` é ajustado dentro de cada pipeline de treino.

In [4]:
raw_feature_columns = ['nr_convenio', 'data_assinatura', 'data_proposta', 'data_fim_vigencia_original', *NUMERIC_FEATURES[:3], *CATEGORICAL_FEATURES]
raw_features = instrumentos[raw_feature_columns].copy()
features, predictor_metadata = select_predictors(raw_features)
with open(OUT / 'dicionario_preditores.json', 'w', encoding='utf-8') as stream:
    json.dump(predictor_metadata, stream, ensure_ascii=False, indent=2)
assert not set(features.columns) & {'valor_desembolsado', 'data_primeiro_desembolso', 'resultado_anulado', 'resultado_rescindido', 'resultado_contas_rejeitadas'}
model_frame = target_frame[['nr_convenio', 'codigo_ibge', 'ano_ref', TARGET, 'censurado', 'dias_ate_primeiro_desembolso', 'resultado_anulado', 'resultado_rescindido', 'resultado_contas_rejeitadas', 'resultado_prorrogado']].join(features)
splits = temporal_split(model_frame)
excluded = pd.DataFrame({'variavel': predictor_metadata['excluded'], 'motivo': 'identificador, alvo, pós-evento, status ou indisponível na assinatura'})
excluded.to_csv(OUT / 'variaveis_excluidas_leakage.csv', index=False)
print({'preditores': FEATURE_COLUMNS, 'excluidas': len(excluded)})

{'preditores': ['valor_global_original', 'valor_global_assinatura', 'valor_contrapartida', 'duracao_original_dias', 'dias_proposta_ate_assinatura', 'modalidade', 'orgao_superior', 'orgao', 'origem_emenda', 'emenda_individual', 'emenda_impositiva'], 'excluidas': 22}

## Coortes, baselines e validação cruzada

O treino é 2022, a validação é 2023 e o teste final é 2024. O tuning usa somente o treino, com `StratifiedGroupKFold`, grupos por município, semente fixa e Average Precision como métrica primária. Dummy prior, Dummy most_frequent e regressão logística são baselines de auditoria; não substituem XGBoost e LightGBM.


In [5]:
train, validation, test = splits[2022], splits[2023], splits[2024]
fold_rows = []
for fold, (train_idx, valid_idx) in enumerate(cv_folds(train[FEATURE_COLUMNS], train[TARGET].astype(int), train.codigo_ibge, 3)):
    fold_rows.append({'fold': fold, 'treino': len(train_idx), 'validacao': len(valid_idx), 'municipios_treino': train.iloc[train_idx].codigo_ibge.nunique(), 'municipios_validacao': train.iloc[valid_idx].codigo_ibge.nunique()})
pd.DataFrame(fold_rows).to_csv(OUT / 'folds.csv', index=False)
print(pd.DataFrame(fold_rows).to_string(index=False))

 fold  treino  validacao  municipios_treino  municipios_validacao
    0     159         80                 86                    43
    1     160         79                 84                    45
    2     159         80                 88                    41


## XGBoost e LightGBM

Apenas os dois modelos atribuídos a Ítalo são treinados. Os seis modelos restantes aparecem no registro como `pendente_integracao`, sem métricas inventadas.

In [6]:
registry = model_registry()
with open(OUT / 'registro_modelos.json', 'w', encoding='utf-8') as stream:
    json.dump(registry, stream, ensure_ascii=False, indent=2, default=str)

results = []
threshold_rows = []
baseline_rows = []
baseline_models = fit_baselines(train[FEATURE_COLUMNS], train[TARGET].astype(int))
for model_name, estimator in baseline_models.items():
    for partition, frame in [('validation', validation), ('test', test)]:
        probabilities = estimator.predict_proba(frame[FEATURE_COLUMNS])[:, 1]
        metric = classification_metrics(frame[TARGET].astype(int), probabilities)
        baseline_rows.append({'modelo': model_name, 'particao': partition, **{k: v for k, v in metric.items() if not isinstance(v, (list, dict))}})

for model_name in ('xgboost', 'lightgbm'):
    search = tune_model(model_name, train[FEATURE_COLUMNS], train[TARGET].astype(int), train.codigo_ibge)
    validation_prob = search.predict_proba(validation[FEATURE_COLUMNS])[:, 1]
    threshold = choose_threshold(validation[TARGET].astype(int), validation_prob, return_audit=True)
    threshold_rows.append({'modelo': model_name, **threshold})
    test_prob = search.predict_proba(test[FEATURE_COLUMNS])[:, 1]
    for partition, frame, probabilities in [('validation', validation, validation_prob), ('test', test, test_prob)]:
        metric = classification_metrics(frame[TARGET].astype(int), probabilities, threshold['threshold'])
        results.append({'modelo': model_name, 'particao': partition, **{k: v for k, v in metric.items() if not isinstance(v, (list, dict))}})
        diagnose_errors(frame, probabilities, threshold['threshold'], model_name).to_csv(OUT / f'erros_{model_name}_{partition}.csv', index=False)
    pd.DataFrame(search.cv_results_).to_csv(OUT / f'cv_{model_name}.csv', index=False)
    with open(OUT / f'hiperparametros_{model_name}.json', 'w', encoding='utf-8') as stream:
        json.dump({'melhores_parametros': search.best_params_, 'average_precision_cv': search.best_score_, 'threshold_validation': threshold}, stream, indent=2, default=str)
    permutation = permutation_sanity(validation[TARGET].astype(int), validation_prob, seed=42)
    with open(OUT / f'sanidade_permutacao_{model_name}.json', 'w', encoding='utf-8') as stream:
        json.dump(permutation, stream, indent=2)

comparison = pd.DataFrame(results)
comparison.to_csv(OUT / 'metricas_modelos.csv', index=False)
pd.DataFrame(baseline_rows).to_csv(OUT / 'metricas_baselines.csv', index=False)
pd.DataFrame(threshold_rows).to_csv(OUT / 'metricas_thresholds.csv', index=False)
print(comparison.to_string(index=False))
print(pd.DataFrame(threshold_rows).to_string(index=False))


  modelo   particao   n  positives  negatives  prevalence  average_precision  average_precision_gain_absolute  average_precision_gain_relative  balanced_accuracy  precision   recall       f1  specificity  precision_at_k  recall_at_k  precision_at_10  recall_at_10  precision_at_20  recall_at_20  precision_at_30  recall_at_30  brier_score  log_loss  threshold  k_fraction  roc_auc
 xgboost validation 386        309         77    0.800518           0.810657                         0.010139                         0.012665           0.581831   0.847222 0.592233 0.697143     0.571429        0.769231     0.194175         0.820513      0.103560         0.769231      0.194175         0.793103      0.297735     0.197528  0.900685   0.989087         0.2 0.548901
 xgboost       test 242        212         30    0.876033           0.901001                         0.024968                         0.028501           0.531289   0.892157 0.429245 0.579618     0.633333        0.857143     0.198113      

## Relação descritiva com resultados administrativos

Os campos `resultado_*` não entram no treinamento. Podem ser cruzados depois da previsão apenas como descrição, sem interpretação causal. O alvo mede atraso financeiro inicial, não conclusão física, qualidade da entrega ou risco geral de contratos públicos.

In [7]:
administrative = target_frame.assign(
    atraso_observavel=target_frame[TARGET].notna(),
)[['ano_ref', TARGET, 'resultado_anulado', 'resultado_rescindido', 'resultado_contas_rejeitadas', 'resultado_prorrogado']]
administrative.to_csv(OUT / 'relacao_desfechos_administrativos.csv', index=False)
limitations = [
    'O horizonte de 365 dias é uma convenção operacional do piloto, não prazo legal.',
    'O alvo mede atraso no início da execução financeira federal.',
    'O alvo não mede conclusão física, qualidade da entrega ou todos os contratos públicos.',
    'Resultados restritos aos instrumentos discricionários observados na Paraíba.',
    'Ausência de desembolso pode ter justificativa administrativa legítima.',
    'Associações com resultado_* são descritivas e não causais.',
    'Os seis modelos dos colegas permanecem pendentes de integração.',
]
(OUT / 'limitacoes.md').write_text('\n'.join(f'- {item}' for item in limitations), encoding='utf-8')
print('Artefatos derivados salvos em', OUT.relative_to(ROOT).as_posix())

Artefatos derivados salvos em .local/e05_italo


## Calibração, permutação e decisão

Average Precision é interpretada contra a prevalência. Brier e Log Loss são apenas métricas de calibração. O teste de sanidade com alvo permutado é reportado separadamente; AP alto em prevalência alta não é evidência suficiente de discriminação. O limiar principal é congelado pela maior balanced accuracy na validação de 2023 e não é recalculado com 2024.


In [8]:
pd.DataFrame(threshold_rows).to_csv(OUT / 'metricas_thresholds.csv', index=False)
print('limiar principal: balanced_accuracy na validação 2023')
print('não há seleção de variáveis, hiperparâmetros ou limiar com o teste 2024')


limiar principal: balanced_accuracy na validação 2023
não há seleção de variáveis, hiperparâmetros ou limiar com o teste 2024
